# Renewal EDA

## Purpose
Train a Renewal model using **flowchart-aligned factors**:
- **M_Health** (health_score)
- **F_Days_Renewal** (days_until_renewal)
- **renewal_stage** (raw) and other renewal-related signals

**Target:** Renewal outcome — binary `will_renew` (renewal_stage == 'renewed' or status == 'renewed') or multi-class renewal_stage. Here we use binary for simplicity.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import joblib
import json

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries OK')

Libraries OK


## 1. Load data

In [2]:
# Resolve path to Research/customer_data_25000.xlsx
cwd = Path.cwd()
research = cwd if (cwd / 'customer_data_25000.xlsx').exists() else (cwd.parent if (cwd.parent / 'customer_data_25000.xlsx').exists() else cwd / 'Research')
if not (research / 'customer_data_25000.xlsx').exists():
    research = Path(r'D:\Projects_Main\Renewal-Upsell-Advisor\Research')
data_path = research / 'customer_data_25000.xlsx'
df = pd.read_excel(data_path, sheet_name='Accounts')
print(f'Loaded: {df.shape[0]:,} rows')
df.head()

Loaded: 25,000 rows


,name,domain,industry,company_size,arr,mrr,contract_start_date,contract_end_date,renewal_date,last_contact_date,...,sentiment_category,licenses_total,licenses_used,utilization_percentage,csm_name,csm_email,primary_contact_name,primary_contact_email,primary_contact_phone,salesforce_id
0,Cole LLC,colellc.com,Technology,Medium,156049,13004.08,2025-06-12,2026-06-12,2026-06-12,2026-02-14 10:30:11,...,negative,20,15,79,Emily Rodriguez,emily.rodriguez@company.com,Danielle Johnson,john21@example.net,001-581-896-0013x3890,SF-835392
1,"Stevens, Martinez and Nielsen",stevensmartinezandnielsen.com,Media,Enterprise,378618,31551.50,2025-05-31,2026-05-31,2026-05-31,2026-02-13 10:30:11,...,very_negative,30,23,79,Sarah Chen,sarah.chen@company.com,Lisa Smith,helenpeterson@example.org,651.216.1559,NaN
2,Clark-Adams,clark-adams.com,Manufacturing,Medium,62823,5235.25,2025-05-06,2026-05-06,2026-05-06,2026-02-07 10:30:11,...,very_positive,150,52,35,Emily Rodriguez,emily.rodriguez@company.com,Christian Carter,barbara10@example.net,441.731.6475,SF-148050
3,"Porter, Wilkerson and Day",porterwilkersonandday.com,Research,Medium,96054,8004.50,2025-06-15,2026-06-15,2026-06-15,2026-02-14 10:30:11,...,very_negative,200,142,71,Sarah Chen,sarah.chen@company.com,Sharon Wong,amandasanchez@example.com,(748)535-0305x6413,SF-356702
4,Carlson-Mcdonald,carlson-mcdonald.com,Consulting,Large,257691,21474.25,2025-04-12,2026-04-12,2026-04-12,2026-01-23 10:30:11,...,very_positive,100,32,32,James Wilson,james.wilson@company.com,Douglas Taylor,julie69@example.com,(332)887-1012x269,NaN


## 2. Flowchart features: health_score, days_until_renewal, renewal_stage (optional)

In [3]:
X = df.copy()
# F_Days_Renewal
X['renewal_date'] = pd.to_datetime(X['renewal_date'], errors='coerce')
ref = X['renewal_date'].max()
if pd.isna(ref):
    ref = pd.Timestamp.now()
X['days_until_renewal'] = (X['renewal_date'] - ref).dt.days
X['days_until_renewal'] = X['days_until_renewal'].fillna(X['days_until_renewal'].median())

feature_cols_num = ['health_score', 'days_until_renewal']
X_num = X[feature_cols_num].fillna(X[feature_cols_num].median())
X_cat = pd.get_dummies(X['renewal_stage'].fillna('t90'), drop_first=True)
X_all = pd.concat([X_num, X_cat], axis=1)
feature_cols = list(X_all.columns)

y = ((df['renewal_stage'] == 'renewed') | (df['status'] == 'renewed')).astype(int)
print('Features (flowchart): health_score, days_until_renewal, renewal_stage (one-hot)')
print('Target: will_renew (0/1)')
print('Class balance:', y.value_counts().to_dict())
X_all.head()

Features (flowchart): health_score, days_until_renewal, renewal_stage (one-hot)
Target: will_renew (0/1)
Class balance: {0: 20931, 1: 4069}


,health_score,days_until_renewal,t30,t60,t90
0,66,-9,False,False,True
1,56,-21,False,False,False
2,61,-46,False,False,True
3,56,-6,False,False,True
4,55,-70,False,True,False


## 3. Train/test split and scale

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42, stratify=y)
scaler = RobustScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print('Train:', X_train_s.shape, 'Test:', X_test_s.shape)

Train: (20000, 5) Test: (5000, 5)


## 4a. Train each model with hyperparameter tuning

Train all base models with `RandomizedSearchCV`; then stack them in the next section.

In [5]:
# Train each model with hyperparameter tuning; store best estimators in `models`
models = {}
results = {}
best_params = {}

# 1. XGBoost
print("1. XGBoost (RandomizedSearchCV)...")
xgb_base = XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='logloss')
xgb_params = {'n_estimators': [150, 200, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, 7], 'subsample': [0.8, 0.9], 'reg_alpha': [0.1, 0.5], 'reg_lambda': [1.0, 2.0]}
xgb_search = RandomizedSearchCV(xgb_base, xgb_params, n_iter=20, cv=5, scoring='f1', n_jobs=-1, random_state=42)
xgb_search.fit(X_train_s, y_train)
models['XGBoost'] = xgb_search.best_estimator_
best_params['XGBoost'] = xgb_search.best_params_
pred = models['XGBoost'].predict(X_test_s)
proba = models['XGBoost'].predict_proba(X_test_s)[:, 1]
results['XGBoost'] = {'accuracy': accuracy_score(y_test, pred), 'f1': f1_score(y_test, pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, proba)}
print(f"   Best params: {xgb_search.best_params_}\n   Acc={results['XGBoost']['accuracy']:.4f}, F1={results['XGBoost']['f1']:.4f}, AUC={results['XGBoost']['roc_auc']:.4f}")

# 2. LightGBM
print("2. LightGBM (RandomizedSearchCV)...")
lgb_base = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)
lgb_params = {'n_estimators': [150, 200, 300], 'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, 7], 'num_leaves': [31, 50], 'subsample': [0.8, 0.9], 'reg_alpha': [0.1, 0.5], 'reg_lambda': [1.0, 2.0]}
lgb_search = RandomizedSearchCV(lgb_base, lgb_params, n_iter=20, cv=5, scoring='f1', n_jobs=-1, random_state=42)
lgb_search.fit(X_train_s, y_train)
models['LightGBM'] = lgb_search.best_estimator_
best_params['LightGBM'] = lgb_search.best_params_
pred = models['LightGBM'].predict(X_test_s)
proba = models['LightGBM'].predict_proba(X_test_s)[:, 1]
results['LightGBM'] = {'accuracy': accuracy_score(y_test, pred), 'f1': f1_score(y_test, pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, proba)}
print(f"   Best params: {lgb_search.best_params_}\n   Acc={results['LightGBM']['accuracy']:.4f}, F1={results['LightGBM']['f1']:.4f}, AUC={results['LightGBM']['roc_auc']:.4f}")

# 3. CatBoost
print("3. CatBoost (RandomizedSearchCV)...")
cat_base = CatBoostClassifier(random_state=42, verbose=False)
cat_params = {'iterations': [150, 200, 300], 'learning_rate': [0.05, 0.1], 'depth': [3, 5, 7], 'subsample': [0.8, 0.9], 'reg_lambda': [1.0, 2.0]}
cat_search = RandomizedSearchCV(cat_base, cat_params, n_iter=15, cv=5, scoring='f1', n_jobs=-1, random_state=42)
cat_search.fit(X_train_s, y_train)
models['CatBoost'] = cat_search.best_estimator_
best_params['CatBoost'] = cat_search.best_params_
pred = models['CatBoost'].predict(X_test_s)
proba = models['CatBoost'].predict_proba(X_test_s)[:, 1]
results['CatBoost'] = {'accuracy': accuracy_score(y_test, pred), 'f1': f1_score(y_test, pred, zero_division=0), 'roc_auc': roc_auc_score(y_test, proba)}
print(f"   Best params: {cat_search.best_params_}\n   Acc={results['CatBoost']['accuracy']:.4f}, F1={results['CatBoost']['f1']:.4f}, AUC={results['CatBoost']['roc_auc']:.4f}")

print("\n--- Model comparison (after hyperparameter tuning) ---")
for name, m in results.items():
    print(f"  {name}: Acc={m['accuracy']:.4f}, F1={m['f1']:.4f}, AUC={m['roc_auc']:.4f}")

1. XGBoost (RandomizedSearchCV)...
   Best params: {'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1}
   Acc=1.0000, F1=1.0000, AUC=1.0000
2. LightGBM (RandomizedSearchCV)...
   Best params: {'subsample': 0.9, 'reg_lambda': 1.0, 'reg_alpha': 0.5, 'num_leaves': 50, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05}
   Acc=1.0000, F1=1.0000, AUC=1.0000
3. CatBoost (RandomizedSearchCV)...
   Best params: {'subsample': 0.8, 'reg_lambda': 1.0, 'learning_rate': 0.1, 'iterations': 150, 'depth': 3}
   Acc=1.0000, F1=1.0000, AUC=1.0000

--- Model comparison (after hyperparameter tuning) ---
  XGBoost: Acc=1.0000, F1=1.0000, AUC=1.0000
  LightGBM: Acc=1.0000, F1=1.0000, AUC=1.0000
  CatBoost: Acc=1.0000, F1=1.0000, AUC=1.0000


## 4b. Stack tuned models

Stack the hyperparameter-tuned base models (XGBoost, LightGBM, CatBoost) with LogisticRegression meta-learner.

In [6]:
# Stack tuned models (separate step)
stacker = StackingClassifier(
    estimators=[('xgb', models['XGBoost']), ('lgb', models['LightGBM']), ('cat', models['CatBoost'])],
    final_estimator=LogisticRegression(max_iter=1000), cv=5
)
stacker.fit(X_train_s, y_train)
pred_s = stacker.predict(X_test_s)
print(f"Stacking: Acc={accuracy_score(y_test, pred_s):.4f}, F1={f1_score(y_test, pred_s, zero_division=0):.4f}")
print(classification_report(y_test, pred_s, target_names=['No Renew', 'Renew']))

Stacking: Acc=1.0000, F1=1.0000
              precision    recall  f1-score   support

    No Renew       1.00      1.00      1.00      4186
       Renew       1.00      1.00      1.00       814

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000



## 5. Save model artifacts (optional)

In [7]:
# Save all trained base models + best model (stacker)
save_dir = Path(r'D:\Projects_Main\Renewal-Upsell-Advisor\Research\Renewal\models')
save_dir.mkdir(parents=True, exist_ok=True)
name_to_file = {'XGBoost': 'renewal_xgb', 'LightGBM': 'renewal_lgb', 'CatBoost': 'renewal_cat'}
for name, model in models.items():
    fname = name_to_file.get(name, name.lower().replace(' ', '_'))
    joblib.dump(model, save_dir / f"{fname}.joblib")
    print(f"Saved: {name} -> {fname}.joblib")
joblib.dump(stacker, save_dir / 'renewal_stacker_best.joblib')
print("Saved: Best model (Stacking) -> renewal_stacker_best.joblib")
joblib.dump(best_params, save_dir / 'renewal_best_params.joblib')
print("Saved: Best hyperparameters -> renewal_best_params.joblib")
joblib.dump(scaler, save_dir / 'renewal_scaler.joblib')
with open(save_dir / 'renewal_feature_names.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)
print("Saved: Scaler and feature names. Location:", save_dir)

Saved: XGBoost -> renewal_xgb.joblib
Saved: LightGBM -> renewal_lgb.joblib
Saved: CatBoost -> renewal_cat.joblib
Saved: Best model (Stacking) -> renewal_stacker_best.joblib
Saved: Best hyperparameters -> renewal_best_params.joblib
Saved: Scaler and feature names. Location: D:\Projects_Main\Renewal-Upsell-Advisor\Research\Renewal\models
